In [5]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [53]:
## Load the train model, scalar pickle, one hot
model=load_model('model.h5')

## load the encoder and scalar

with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_enc_geo=pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_enc_gender=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)



In [55]:
## Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [57]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [58]:
## label encoder Gender
input_df['Gender'] = label_enc_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [59]:
## One hot encoding Geography
onehotenc_geo = onehot_enc_geo.transform([input_df['Geography']]).toarray()
onehotenc_geo_df = pd.DataFrame(onehotenc_geo,columns=onehot_enc_geo.get_feature_names_out(['Geography']))
onehotenc_geo_df

/Users/rama/ANN Classification/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [60]:
## combine one hot encoded columns with input data
input_df = pd.concat([input_df.drop('Geography',axis=1),onehotenc_geo_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [61]:
## Scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.52807513,  0.91023746,  0.10509787, -0.69811855, -0.26546391,
         0.80030876,  0.64406831,  0.96921579, -0.87689367,  0.99443217,
        -0.57921062, -0.57119169]])

In [62]:
## Predict churn

prediction=model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 32ms/step


array([[0.03514627]], dtype=float32)

In [63]:
prediction_proba = prediction[0][0]
prediction_proba

0.03514627

In [64]:
if prediction_proba > 0.5 :
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
